# Phase 2 — Query Transformations & Context Construction

## 1. Objective

Extend the completed Basic RAG pipeline without replacing it. This experiment evaluates deterministic query variants, configurable top-10 multi-query retrieval, exact evidence deduplication, neighbor expansion, overlap merging, bounded context construction, metadata-rich citations, and explicit safe failure.

All processing remains local and offline. Notebook 01 is a frozen Phase 1 milestone.

## 2. Theory

A single phrasing may not align with the vocabulary used in an enterprise document. Phase 2 creates inspectable variants of the same intent and retrieves evidence for each variant. It merges chunks—not generated answers—then removes duplicates using `(source, page, chunk_id)`.

Retrieved chunks can lack the surrounding sentence needed to interpret a rule. Configurable neighbor expansion adds adjacent source chunks. Contiguous chunks are merged with splitter overlap removed, then the final context is compressed to a deterministic character budget before local generation.

## 3. Architecture

```text
User query
  → QueryTransformer (original / rewritten / keyword / domain)
  → Multi-query semantic retrieval (configurable top-k per query)
  → Evidence merge → exact deduplication
  → Neighbor expansion → overlap merge → context compression
  → [future reranker extension point]
  → Grounded prompt → local LLM → metadata-rich citations
```

`Phase2RAGPipeline` subclasses `BasicRAGPipeline`; ingestion, chunking, embeddings, Qdrant storage, and the local model abstraction are reused unchanged.

## 4. Implementation

### 4.1 Setup and configuration

The configuration keeps Phase 1's `top_k=3` intact and introduces the Phase 2 retrieval depth separately.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from IPython.display import display

from cial_knowledge_os import (
    ContextBuilder,
    Phase2Config,
    Phase2RAGPipeline,
    QueryTransformer,
    batch_retrieval_trace_table,
    citation_quality_table,
    context_stage_counts_table,
    duplicate_chunk_frequency_table,
    export_batch_answers,
    neighbor_expansion_table,
    plot_context_stage_counts,
    plot_duplicate_chunk_frequency,
    plot_retrieval_comparison,
    plot_retrieval_scores,
    print_retrieval_results,
    query_variants_table,
    render_citations,
    retrieval_chunks_table,
    retrieval_comparison_table,
    search_similar_chunks,
)

config = Phase2Config(
    project_root=PROJECT_ROOT,
    retrieval_top_k=10,
    enable_query_rewrite=True,
    enable_keyword_expansion=True,
    enable_domain_reformulation=True,
    enable_multi_query=True,
    enable_neighbor_expansion=True,
    neighbor_window=1,
    enable_overlap_merging=True,
    enable_context_compression=True,
    max_context_chars=3_000,
)
{
    "phase_1_top_k_unchanged": config.top_k,
    "phase_2_retrieval_top_k": config.retrieval_top_k,
    "neighbor_window": config.neighbor_window,
    "max_context_chars": config.max_context_chars,
}

### 4.2 Query transformations demonstrated independently

These strategies are deterministic and require no network or LLM call. A future local rewrite model can be registered through the same strategy interface.

In [ ]:
transformer = QueryTransformer(config)
transformation_question = "Could you please explain runway maintenance safety requirements?"

demonstration_variants = [
    transformer.transform(transformation_question, technique)
    for technique in (
        "original",
        "rewritten",
        "keyword_expanded",
        "domain_reformulation",
    )
]
display(query_variants_table(demonstration_variants))

### 4.3 Data loading, processing, and local indexing

The Phase 1 methods are reused. The embedding model and Ollama model must already be installed locally; this notebook never downloads a model.

In [ ]:
pipeline = Phase2RAGPipeline(config=config)
documents = pipeline.load()
chunks = pipeline.chunk()
embeddings = pipeline.embed()
client = pipeline.index()

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")
print(f"Embedding shape: {embeddings.shape}")

### 4.4 Before vs after retrieval

The baseline is the original query with Phase 1 depth 3. The Phase 2 result retrieves up to 10 chunks per distinct query variant and merges duplicate evidence before returning it.

In [ ]:
question = "What safety steps are required before electrical maintenance?"

baseline_results = search_similar_chunks(
    pipeline.client,
    question,
    pipeline.embedding_model,
    config,
    top_k=config.top_k,
)
phase2_results = pipeline.retrieve(question)

print("PHASE 1 BASELINE — original query, top-3")
print_retrieval_results(baseline_results)
print("\nPHASE 2 — transformed queries, top-10 per query, deduplicated")
print_retrieval_results(phase2_results)
print("\nQuery variants:")
for variant in pipeline.last_query_variants:
    print(variant.as_dict())

display(query_variants_table(pipeline.last_query_variants))
display(retrieval_comparison_table(baseline_results, phase2_results))
plot_retrieval_comparison(baseline_results, phase2_results)

### 4.5 Paraphrase robustness and comparison questions

The same retrieval pipeline is evaluated with different phrasings and with a question that requires evidence from more than one document.

In [ ]:
evaluation_queries = {
    "direct": "How often must runway surface inspections occur?",
    "paraphrase": "What is the required interval for checking the runway surface?",
    "comparison": "Compare runway entry controls with energized-panel maintenance controls.",
    "enterprise": "Summarize mandatory actions, responsible teams, and escalation conditions for operational safety inspections.",
}

retrieval_evaluation = {}
for label, evaluation_query in evaluation_queries.items():
    retrieved = pipeline.retrieve(evaluation_query)
    retrieval_evaluation[label] = retrieved
    top_sources = [result["source"] for result in retrieved[:3]]
    print(f"{label:>12}: {len(retrieved)} unique chunks | top sources={top_sources}")

### 4.6 Context construction stages

Every stage is returned for inspection. Metadata is visible before citations, context formatting, and generation.

In [ ]:
enterprise_query = evaluation_queries["enterprise"]
retrieved = pipeline.retrieve(enterprise_query)
context_result = ContextBuilder(config).build(
    pipeline.last_merged_retrieval,
    corpus_chunks=pipeline.chunks,
)

print("Stage counts:", context_result.stage_counts())
for stage_name in ("retrieved", "deduplicated", "expanded", "merged", "compressed"):
    stage = getattr(context_result, stage_name)
    print(f"\n{stage_name.upper()} ({len(stage)})")
    print_retrieval_results(stage[:5])

print("\nBEFORE DEDUPLICATION")
display(retrieval_chunks_table(context_result.retrieved, stage="retrieved").head(20))
print("AFTER DEDUPLICATION")
display(retrieval_chunks_table(context_result.deduplicated, stage="deduplicated").head(20))
print("DUPLICATE FREQUENCY BY (SOURCE, PAGE, CHUNK_ID)")
display(duplicate_chunk_frequency_table(context_result.retrieved).head(20))
print("NEIGHBOR EXPANSION PROVENANCE")
display(neighbor_expansion_table(context_result.deduplicated, context_result.expanded).head(30))
print("CONTEXT CONSTRUCTION COUNTS")
display(context_stage_counts_table(context_result))

print("\nFINAL COMPRESSED CONTEXT\n")
print(context_result.context)

### 4.7 Generation and citation quality

Only one answer is generated from merged evidence. Citations correspond to the compressed blocks actually sent to the local model.

In [ ]:
response = pipeline.answer(enterprise_query)
print(response["answer"])
print("\nCitation metadata:")
print(render_citations(response["citations"]))
display(citation_quality_table(response["citations"]))
print("\nInspectable stage counts:", response["stage_counts"])

## 5. Visualization

These reusable plots consume the same real trace objects shown above. They expose Phase 1 vs Phase 2 retrieval volume, duplicate pressure across transformed queries, similarity-score distribution, and evidence expansion/reduction through final context construction.

In [ ]:
plot_retrieval_scores(phase2_results)
plot_retrieval_comparison(baseline_results, phase2_results)
plot_duplicate_chunk_frequency(context_result.retrieved)
plot_context_stage_counts(context_result)

## 6. Benchmark

Inspect local latency, context size, evidence reduction, and citation completeness. Retrieval cost grows with the number of distinct variants, so quality gains must be evaluated against latency.

In [ ]:
citation_completeness = all(
    citation.get("source_file")
    and citation.get("chunk_id")
    and citation.get("score") is not None
    for citation in response["citations"]
)

{
    "metrics_seconds": {key: round(value, 4) for key, value in pipeline.metrics.items()},
    "query_variant_count": len(response["query_variants"]),
    "context_characters": len(response["context"]),
    "context_budget": config.max_context_chars,
    "citation_count": len(response["citations"]),
    "citation_fields_complete": citation_completeness,
}

### 6.1 Phase 2 batch evaluation, CSV export, and retrieval traces

The shared exporter calls `pipeline.answer()` for every question, so each row uses query transformations, multi-query top-10 retrieval, deduplication, neighbor expansion, context construction, one grounded local-model answer, and citation formatting. The final table reads the real exported `retrieval_trace` column.

In [ ]:
batch_evaluation_questions = [
    # Factual
    "How frequently must routine runway surface inspections be performed?",
    # Comparison
    "Compare runway-entry controls with controls for work on energized electrical panels.",
    # Executive summary
    "Provide an executive summary of the operational safety controls in the indexed SOPs.",
    # Enterprise operational
    "What mandatory actions, responsible teams, and escalation conditions govern airport safety inspections?",
    # Paraphrase
    "When passenger lines grow beyond their assigned space, what should airport staff do?",
    # Unsupported / safe-failure case
    "What dishes are listed on tomorrow's staff cafeteria menu?",
]

phase2_csv_path = export_batch_answers(
    pipeline=pipeline,
    questions=batch_evaluation_questions,
    run_name="02_Query_Transformations_and_Context_Construction",
)
print(f"Phase 2 CSV exported to: {phase2_csv_path}")

In [ ]:
batch_trace = batch_retrieval_trace_table(phase2_csv_path)
display(batch_trace)
assert "retrieval_trace" in batch_trace.columns

## 7. Advantages

- Better vocabulary and paraphrase coverage without cloud services.
- Higher configurable recall while keeping the final prompt bounded.
- Exact duplicate removal occurs before citations and generation.
- Neighbor evidence improves continuity around retrieved passages.
- Every intermediate query and context stage remains inspectable.
- Phase 1 APIs and defaults remain available unchanged.

## 8. Limitations

- Deterministic transformations cannot infer every domain synonym.
- Multi-query retrieval increases embedding and vector-search latency.
- Neighbor expansion can introduce irrelevant nearby text.
- Character-based compression can truncate a final block; token-aware compression is a future refinement.
- Similarity scores from separate query variants are merged directly and are not calibrated or reranked.

## 9. Enterprise Considerations

All documents, embeddings, queries, prompts, and generated answers remain on the local host. Rich metadata is retained for auditability. Future access-control filters must be applied during retrieval before neighbor expansion so an authorized seed cannot expand into an unauthorized chunk. Production evaluation should measure recall, citation accuracy, latency, token usage, duplicate-document behavior, missing pages, conflicting versions, and safe failure.

## 10. What we'll improve in the next notebook

Notebook 03 can add keyword/BM25 retrieval through the existing retrieval boundary and merge it with semantic evidence. Notebook 04 can insert a local reranker between merged retrieval and context construction. Notebook 05 can reuse the inspectable pipeline state for bounded planner or verifier roles. Those features are extension points only and are not implemented here.

In [ ]:
# Release the embedded Qdrant file lock when the experiment is complete.
pipeline.close()